# QShield: Classical Machine Learning Baselines

In [2]:
# Import classical machine learning libraries and evaluation metrics

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

In [3]:
# Load balanced fraud dataset created during exploratory analysis

df = pd.read_csv(
    "/Users/srinidhinunna/Desktop/HighQ/data/balanced_fraud_dataset.csv"
)

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (16426, 14)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_error_orig,balance_error_dest,amount_ratio
0,139,1,97502.58,C378583051,0.00,0.00,C2024388240,143669.50,241172.08,0,0,-9.750258e+04,2.910383e-11,97502.580000
1,131,0,354402.90,C558895391,410027.06,55624.16,C810756973,1314099.04,1668501.94,0,0,-2.910383e-11,0.000000e+00,0.864338
2,98,1,8055.06,C1544350298,8055.06,0.00,C912405348,0.00,8055.06,1,0,0.000000e+00,0.000000e+00,0.999876
3,550,0,342309.91,C662184778,342309.91,0.00,C1740503020,0.00,0.00,1,0,0.000000e+00,3.423099e+05,0.999997
4,586,1,2581549.92,C648614053,2581549.92,0.00,C213455810,0.00,2581549.92,1,0,0.000000e+00,0.000000e+00,1.000000


## Feature Selection

In [5]:
# Select fraud-relevant features for model training

features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "balance_error_orig",
    "balance_error_dest",
    "amount_ratio",
    "type"
]

X = df[features]

y = df["isFraud"]

print(X.shape)

(16426, 9)


## Train-Test Split

In [7]:
# Create stratified training and testing datasets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (13140, 9)
Test : (3286, 9)


## Model Configuration

In [9]:
# Define classical machine learning baseline models

models = {
    
    "Logistic Regression":
        LogisticRegression(
            max_iter=1000
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ),

    "RBF SVM":
        SVC(
            kernel="rbf",
            probability=True
        ),

    "XGBoost":
        XGBClassifier(
            eval_metric="logloss",
            random_state=42
        )
}

## Model Training and Evaluation

In [11]:
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.svm import SVC

# Replace SVC(kernel="rbf", probability=True) with CalibratedClassifierCV
models["RBF SVM"] = CalibratedClassifierCV(
    estimator=SVC(kernel="rbf", random_state=42),
    ensemble=False,
    cv=5,
)

results = []

for model_name, model in models.items():
    print("=" * 60)
    print(f"Training {model_name}")

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X_test)[:, 1]
    else:
        scores = model.decision_function(X_test)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, scores),
    })

print("=" * 60)
print("Finished")

metrics_df = (
    pd.DataFrame(results)
    .sort_values(by="ROC_AUC", ascending=False)
    .reset_index(drop=True)
)

display(metrics_df.round(4))

Training Logistic Regression
Training Random Forest
Training RBF SVM
Training XGBoost
Finished


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Random Forest,0.9997,0.9994,1.0000,0.9997,1.0000
1,XGBoost,0.9994,0.9994,0.9994,0.9994,1.0000
2,Logistic Regression,0.9078,0.9351,0.8764,0.9048,0.9768
3,RBF SVM,0.9047,0.8958,0.9160,0.9058,0.9696


## Baseline Results

In [13]:
# Generate comparison table across all classical models

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "F1",
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
1,Random Forest,0.999696,0.999392,1.000000,0.999696,1.000000
3,XGBoost,0.999391,0.999391,0.999391,0.999391,0.999994
2,RBF SVM,0.904747,0.895833,0.916007,0.905808,0.969609
0,Logistic Regression,0.907791,0.935065,0.876446,0.904807,0.976755


In [14]:
# Export classical model results for final reporting

results_df.to_csv(
    "/Users/srinidhinunna/Desktop/HighQ/data/classical_results.csv",
    index=False
)

print("Results saved successfully")

Results saved successfully


# Key Findings

1. Multiple classical machine learning algorithms were evaluated.
2. Precision, Recall, F1 and ROC-AUC were used instead of Accuracy alone.
3. The strongest performing classical model will serve as the benchmark against the QSVM.
4. These results establish a fair comparison framework for quantum machine learning experiments.